# NSGA-II Multi-objective Optimization with Bias Application

## What is Multi-objective Optimization?

Multi-objective optimization involves optimizing multiple objective functions simultaneously. Unlike single-objective optimization, there's usually no single optimal solution, but rather a set of solutions called the Pareto Front.

## Introduction to NSGA-II Algorithm

NSGA-II (Non-dominated Sorting Genetic Algorithm II) is a classic multi-objective optimization algorithm with these features:
- Fast non-dominated sorting
- Crowding distance calculation for diversity preservation
- Elitism preservation strategy

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import random
from typing import List, Tuple

# Import NSGABLACK components
import sys
sys.path.append('../..')
from solvers.nsga2 import NSGA2
from bias.bias_v2 import BiasSystemV2

print("Environment setup complete")
print(f"NumPy version: {np.__version__}")

## Define Multi-objective Test Problems

In [ ]:
# Define multi-objective test problems

class ZDT1:
    """ZDT1 test problem - Classic 2-objective test problem"""
    
    def __init__(self, n_var=30):
        self.n_var = n_var
        self.n_obj = 2
        self.lb = np.zeros(n_var)
        self.ub = np.ones(n_var)
        
    def evaluate(self, x):
        """Evaluate solution x
        
        Args:
            x: Decision variable array
        
        Returns:
            Objective values list [f1, f2]
        """
        x = np.array(x)
        f1 = x[0]
        
        g = 1 + 9 * np.sum(x[1:]) / (self.n_var - 1)
        
        h = 1 - np.sqrt(f1 / g)
        
        f2 = g * h
        
        return [f1, f2]

class ZDT2:
    """ZDT2 test problem - Concave Pareto front"""
    
    def __init__(self, n_var=30):
        self.n_var = n_var
        self.n_obj = 2
        self.lb = np.zeros(n_var)
        self.ub = np.ones(n_var)
    
    def evaluate(self, x):
        x = np.array(x)
        f1 = x[0]
        
        g = 1 + 9 * np.sum(x[1:]) / (self.n_var - 1)
        
        h = 1 - (f1 / g)**2
        
        f2 = g * h
        
        return [f1, f2]

# Create test problems
print("Creating multi-objective test problems:")
print("-"*40)

zdt1 = ZDT1(n_var=10)
zdt2 = ZDT2(n_var=10)

print(f"ZDT1: {zdt1.n_var} variables, {zdt1.n_obj} objectives")
print(f"ZDT2: {zdt2.n_var} variables, {zdt2.n_obj} objectives")

# Test evaluation function
test_x = [0.5] + [0.5] * 9
print(f"\nTest solution: {test_x[:3]}...")
print(f"ZDT1 objectives: {zdt1.evaluate(test_x)}")
print(f"ZDT2 objectives: {zdt2.evaluate(test_x)}")

## NSGA-II with NSGABLACK Framework

In [ ]:
# Test NSGA-II with NSGABLACK framework
print("Testing NSGA-II with NSGABLACK:")
print("-"*40)

# Create problem
problem = zdt1

# Create NSGA-II solver
solver = NSGA2(
    problem=problem,
    pop_size=100,
    max_gen=100
)

# Run optimization
pareto_front = solver.optimize()

# Extract objectives
objectives = np.array([ind.objectives for ind in pareto_front])
print(f"\nPareto front contains {len(pareto_front)} solutions")
print(f"Objective ranges:")
print(f"  f1: [{objectives[:, 0].min():.4f}, {objectives[:, 0].max():.4f}]")
print(f"  f2: [{objectives[:, 1].min():.4f}, {objectives[:, 1].max():.4f}]")

## Integrating Bias with NSGA-II

In [ ]:
# Create a bias system for NSGA-II
bias_system = BiasSystemV2(
    algorithmic_bias=True,
    domain_bias=False,
    adaptive_bias=True
)

print(f"Bias system created: {bias_system}")
print(f"Algorithmic bias weight: {bias_system.algorithmic_weight}")
print(f"Adaptive bias weight: {bias_system.adaptive_weight}")

In [ ]:
# NSGA-II with bias integration
class NSGA2WithBias:
    """NSGA-II with integrated bias system"""
    
    def __init__(self, problem, bias_system=None, pop_size=100, max_gen=100):
        self.problem = problem
        self.bias_system = bias_system
        self.pop_size = pop_size
        self.max_gen = max_gen
        self.population = []
        self.history = []
        
        print(f"Creating NSGA-II with bias")
        print(f"  Population size: {pop_size}")
        print(f"  Max generations: {max_gen}")
        print(f"  Bias system: {'Enabled' if bias_system else 'Disabled'}")
    
    def initialize_population(self):
        """Initialize population"""
        self.population = []
        for _ in range(self.pop_size):
            # Create individual
            decision = np.random.uniform(0, 1, self.problem.n_var)
            objectives = self.problem.evaluate(decision)
            
            individual = type('Individual', (), {
                'decision': decision,
                'objectives': objectives,
                'rank': 0,
                'crowding_distance': 0
            })()
            
            self.population.append(individual)
    
    def apply_bias_to_population(self, generation):
        """Apply bias to guide the population"""
        if self.bias_system is None:
            return
        
        # Calculate population statistics
        objectives = np.array([ind.objectives for ind in self.population])
        avg_fitness = np.mean(objectives, axis=0)
        
        # Adaptive bias strength based on generation
        bias_strength = 0.1 * (1 - generation / self.max_gen)
        
        # Apply bias to decision variables
        for ind in self.population:
            # Bias towards better solutions
            if ind.rank == 0:  # Non-dominated solutions
                # Preserve elite solutions
                bias_factor = 1.0
            else:
                # Guide dominated solutions
                bias_factor = 1 - bias_strength * ind.rank * 0.1
                
            # Apply bias to decision variables
            ind.decision = ind.decision * bias_factor
            ind.decision = np.clip(ind.decision, 0, 1)
            
            # Re-evaluate objectives
            ind.objectives = self.problem.evaluate(ind.decision)
    
    def fast_non_dominated_sort(self):
        """Fast non-dominated sorting"""
        fronts = [[]]
        
        # Calculate domination relationships
        for ind in self.population:
            ind.dominated_count = 0
            ind.dominates = []
            
            for other in self.population:
                if self.dominates(ind, other):
                    ind.dominates.append(other)
                elif self.dominates(other, ind):
                    ind.dominated_count += 1
            
            if ind.dominated_count == 0:
                ind.rank = 0
                fronts[0].append(ind)
        
        # Create other fronts
        i = 0
        while fronts[i]:
            next_front = []
            
            for ind in fronts[i]:
                for other in ind.dominates:
                    other.dominated_count -= 1
                    if other.dominated_count == 0:
                        other.rank = i + 1
                        next_front.append(other)
            
            i += 1
            fronts.append(next_front)
        
        return fronts[:-1]  # Remove last empty front
    
    def dominates(self, ind1, ind2):
        """Check if ind1 dominates ind2"""
        at_least_one_better = False
        
        for i in range(len(ind1.objectives)):
            if ind1.objectives[i] > ind2.objectives[i]:
                return False
            if ind1.objectives[i] < ind2.objectives[i]:
                at_least_one_better = True
        
        return at_least_one_better
    
    def calculate_crowding_distance(self, front):
        """Calculate crowding distance"""
        if len(front) == 0:
            return
        
        n_obj = len(front[0].objectives)
        n_ind = len(front)
        
        # Initialize crowding distance
        for ind in front:
            ind.crowding_distance = 0
        
        # Calculate crowding distance for each objective
        for m in range(n_obj):
            # Sort by m-th objective
            front.sort(key=lambda x: x.objectives[m])
            
            # Boundary points set to infinity
            front[0].crowding_distance = float('inf')
            front[-1].crowding_distance = float('inf')
            
            # Calculate crowding distance for middle points
            obj_min = front[0].objectives[m]
            obj_max = front[-1].objectives[m]
            
            if obj_max - obj_min > 1e-10:
                for i in range(1, n_ind - 1):
                    distance = front[i+1].objectives[m] - front[i-1].objectives[m]
                    front[i].crowding_distance += distance / (obj_max - obj_min)
    
    def optimize(self):
        """Execute optimization"""
        print(f"\nStarting NSGA-II with bias optimization...")
        
        # Initialize population
        self.initialize_population()
        print(f"Initial population generated, size: {len(self.population)}")
        
        # Evolution loop
        for generation in range(self.max_gen):
            # Apply bias
            self.apply_bias_to_population(generation)
            
            # Non-dominated sorting
            fronts = self.fast_non_dominated_sort()
            
            # Calculate crowding distance
            for front in fronts:
                self.calculate_crowding_distance(front)
            
            # Environmental selection (simplified)
            new_population = []
            for front in fronts:
                if len(new_population) + len(front) <= self.pop_size:
                    new_population.extend(front)
                else:
                    front.sort(key=lambda x: x.crowding_distance, reverse=True)
                    remaining = self.pop_size - len(new_population)
                    new_population.extend(front[:remaining])
                    break
            
            self.population = new_population
            
            # Record history
            front = [ind for ind in self.population if ind.rank == 0]
            self.history.append(front)
            
            # Output progress
            if (generation + 1) % 20 == 0:
                print(f"  Generation: {generation + 1}/{self.max_gen}, First front size: {len(front)}")
        
        print(f"\nOptimization complete!")
        print(f"Final first front size: {len([ind for ind in self.population if ind.rank == 0])}")
        
        # Return Pareto front
        pareto_front = [ind for ind in self.population if ind.rank == 0]
        return pareto_front

print("NSGA-II with bias integration class defined")

## Comparing NSGA-II With and Without Bias

In [ ]:
# Compare NSGA-II with and without bias
print("\nComparing NSGA-II with and without bias:")
print("-"*40)

# Run without bias
print("\nRunning NSGA-II WITHOUT bias...")
nsga2_no_bias = NSGA2WithBias(
    problem=zdt1,
    bias_system=None,
    pop_size=100,
    max_gen=100
)
pareto_no_bias = nsga2_no_bias.optimize()

# Run with bias
print("\nRunning NSGA-II WITH bias...")
nsga2_with_bias = NSGA2WithBias(
    problem=zdt1,
    bias_system=bias_system,
    pop_size=100,
    max_gen=100
)
pareto_with_bias = nsga2_with_bias.optimize()

# Extract objectives for comparison
obj_no_bias = np.array([ind.objectives for ind in pareto_no_bias])
obj_with_bias = np.array([ind.objectives for ind in pareto_with_bias])

print(f"\nResults comparison:")
print(f"Without bias: {len(pareto_no_bias)} solutions")
print(f"With bias: {len(pareto_with_bias)} solutions")

In [ ]:
# Generate true Pareto front for comparison
def true_pareto_front_zdt1(n_points=100):
    f1 = np.linspace(0, 1, n_points)
    f2 = 1 - np.sqrt(f1)
    return f1, f2

true_f1, true_f2 = true_pareto_front_zdt1()

# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Without bias - Pareto front
axes[0, 0].scatter(obj_no_bias[:, 0], obj_no_bias[:, 1], 
                  alpha=0.7, s=50, label='Found solutions', c='blue')
axes[0, 0].plot(true_f1, true_f2, 'r--', linewidth=2, label='True Pareto front')
axes[0, 0].set_xlabel('f1')
axes[0, 0].set_ylabel('f2')
axes[0, 0].set_title(f'NSGA-II WITHOUT Bias ({len(pareto_no_bias)} solutions)')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim([0, 1])
axes[0, 0].set_ylim([0, 1])

# With bias - Pareto front
axes[0, 1].scatter(obj_with_bias[:, 0], obj_with_bias[:, 1], 
                  alpha=0.7, s=50, label='Found solutions', c='green')
axes[0, 1].plot(true_f1, true_f2, 'r--', linewidth=2, label='True Pareto front')
axes[0, 1].set_xlabel('f1')
axes[0, 1].set_ylabel('f2')
axes[0, 1].set_title(f'NSGA-II WITH Bias ({len(pareto_with_bias)} solutions)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xlim([0, 1])
axes[0, 1].set_ylim([0, 1])

# Convergence history - Without bias
front_sizes_no_bias = [len(front) for front in nsga2_no_bias.history]
axes[1, 0].plot(range(len(front_sizes_no_bias)), front_sizes_no_bias, 
               'b-', linewidth=2, label='Without bias')
axes[1, 0].set_xlabel('Generation')
axes[1, 0].set_ylabel('First front size')
axes[1, 0].set_title('Convergence Process - Without Bias')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Convergence history - With bias
front_sizes_with_bias = [len(front) for front in nsga2_with_bias.history]
axes[1, 1].plot(range(len(front_sizes_with_bias)), front_sizes_with_bias, 
               'g-', linewidth=2, label='With bias')
axes[1, 1].set_xlabel('Generation')
axes[1, 1].set_ylabel('First front size')
axes[1, 1].set_title('Convergence Process - With Bias')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate performance metrics
def generational_distance(front, true_front):
    """Calculate Generational Distance (GD)"""
    gd_sum = 0
    for point in front:
        min_dist = float('inf')
        for true_point in true_front:
            dist = np.sqrt((point[0] - true_point[0])**2 + (point[1] - true_point[1])**2)
            if dist < min_dist:
                min_dist = dist
        gd_sum += min_dist**2
    return np.sqrt(gd_sum / len(front))

true_front_array = np.column_stack((true_f1, true_f2))
gd_no_bias = generational_distance(obj_no_bias, true_front_array)
gd_with_bias = generational_distance(obj_with_bias, true_front_array)

print(f"\nPerformance Metrics:")
print(f"Generational Distance (GD) - Without bias: {gd_no_bias:.4f}")
print(f"Generational Distance (GD) - With bias: {gd_with_bias:.4f}")
print(f"\nLower GD values indicate better performance (ideal: 0)")

## Decision Maker Preference Integration

In practical applications, decision makers may have specific preferences. We can select final solutions from the Pareto front based on these preferences.

In [ ]:
# Decision maker preference selection
def select_with_preference(pareto_front, preference_type="utopian_point", preference_params=None):
    """Select solution based on preferences
    
    Args:
        pareto_front: Pareto front
        preference_type: Preference type
        preference_params: Preference parameters
    
    Returns:
        Selected solution
    """
    
    if preference_type == "utopian_point":
        # Utopian point method: select solution closest to ideal point
        if preference_params is None:
            utopian = [0, 0]  # Default utopian point
        else:
            utopian = preference_params
        
        min_dist = float('inf')
        best_solution = None
        
        for ind in pareto_front:
            dist = np.sqrt((ind.objectives[0] - utopian[0])**2 + 
                           (ind.objectives[1] - utopian[1])**2)
            if dist < min_dist:
                min_dist = dist
                best_solution = ind
        
        return best_solution
    
    elif preference_type == "weighted_sum":
        # Weighted sum method
        if preference_params is None:
            weights = [0.5, 0.5]  # Default equal weights
        else:
            weights = preference_params
        
        min_score = float('inf')
        best_solution = None
        
        for ind in pareto_front:
            score = weights[0] * ind.objectives[0] + weights[1] * ind.objectives[1]
            if score < min_score:
                min_score = score
                best_solution = ind
        
        return best_solution
    
    else:
        # Default: select middle solution
        return pareto_front[len(pareto_front)//2]

# Demonstrate preference selection
print("\nDecision maker preference selection demonstration:")
print("-"*40)

# Use the ZDT1 Pareto front obtained with bias
pareto_front = pareto_with_bias

# 1. Utopian point method
utopian_solution = select_with_preference(pareto_front, 
                                        "utopian_point", [0.2, 0.2])
print(f"\nUtopian point [0.2, 0.2] selected solution: {utopian_solution.objectives}")

# 2. Weighted sum method (prefer f1)
weighted_solution = select_with_preference(pareto_front, 
                                        "weighted_sum", [0.8, 0.2])
print(f"Weighted sum [0.8, 0.2] selected solution: {weighted_solution.objectives}")

# 3. Weighted sum method (prefer f2)
weighted_solution2 = select_with_preference(pareto_front, 
                                         "weighted_sum", [0.2, 0.8])
print(f"Weighted sum [0.2, 0.8] selected solution: {weighted_solution2.objectives}")

# Visualize preference selection
objectives = np.array([ind.objectives for ind in pareto_front])

plt.figure(figsize=(10, 8))

# Plot Pareto front
plt.scatter(objectives[:, 0], objectives[:, 1], alpha=0.5, s=30, label='Pareto front')

# Plot selected solutions
plt.scatter(utopian_solution.objectives[0], utopian_solution.objectives[1], 
           s=200, c='red', marker='*', label='Utopian point selection')
plt.scatter(weighted_solution.objectives[0], weighted_solution.objectives[1], 
           s=200, c='green', marker='s', label='Weighted selection[0.8,0.2]')
plt.scatter(weighted_solution2.objectives[0], weighted_solution2.objectives[1], 
           s=200, c='blue', marker='^', label='Weighted selection[0.2,0.8]')

# Plot utopian point
plt.scatter(0.2, 0.2, s=100, c='red', marker='x', linewidth=3)

# Add reference lines
plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.5)
plt.axvline(x=0.2, color='red', linestyle='--', alpha=0.5)

plt.xlabel('f1')
plt.ylabel('f2')
plt.title('Preference-based Solution Selection')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim([0, 1.1])
plt.ylim([0, 1.1])

plt.show()

print("\nPreference selection methods:")
print("1. Utopian point: Select solution closest to ideal point")
print("2. Weighted sum: Select solution based on weight preferences")
print("3. In practice, choose appropriate method based on decision maker needs")

## Advanced Bias Strategies for Multi-objective Optimization

In [ ]:
# Advanced bias strategies for multi-objective optimization
class MultiObjectiveBiasStrategy:
    """Advanced bias strategy for multi-objective optimization"""
    
    def __init__(self):
        self.reference_point = None
        self.preference_direction = None
        self.bias_history = []
        
    def set_reference_point(self, reference_point):
        """Set reference point for bias direction"""
        self.reference_point = np.array(reference_point)
        print(f"Reference point set: {self.reference_point}")
        
    def set_preference_direction(self, direction):
        """Set preference direction for bias"""
        self.preference_direction = np.array(direction) / np.linalg.norm(direction)
        print(f"Preference direction set: {self.preference_direction}")
        
    def apply_region_bias(self, population, target_region="knee"):
        """Apply bias to specific regions of the Pareto front"""
        objectives = np.array([ind.objectives for ind in population])
        
        if target_region == "knee":
            # Identify knee points (points with maximum marginal utility)
            knee_scores = self._calculate_knee_scores(objectives)
            bias_strength = knee_scores / np.max(knee_scores)
            
        elif target_region == "compromise":
            # Bias toward compromise solutions (balanced objectives)
            normalized_obj = objectives / np.max(objectives, axis=0)
            bias_strength = 1 - np.std(normalized_obj, axis=1)
            
        elif target_region == "extreme":
            # Bias toward extreme solutions
            bias_strength = np.max(objectives / np.max(objectives, axis=0), axis=1)
            
        else:
            bias_strength = np.ones(len(population))
            
        # Apply bias
        for i, ind in enumerate(population):
            if bias_strength[i] > 0.7:  # Strong bias for preferred solutions
                # Preserve or improve these solutions
                ind.decision = ind.decision * (1 + 0.05 * np.random.randn(len(ind.decision)))
                ind.decision = np.clip(ind.decision, 0, 1)
                ind.objectives = zdt1.evaluate(ind.decision)
                
        return bias_strength
    
    def _calculate_knee_scores(self, objectives):
        """Calculate knee scores for solutions"""
        if len(objectives) < 3:
            return np.ones(len(objectives))
            
        # Sort by first objective
        sorted_indices = np.argsort(objectives[:, 0])
        sorted_obj = objectives[sorted_indices]
        
        knee_scores = np.zeros(len(objectives))
        
        # Calculate angle for each point
        for i in range(1, len(sorted_obj) - 1):
            v1 = sorted_obj[i-1] - sorted_obj[i]
            v2 = sorted_obj[i+1] - sorted_obj[i]
            
            # Normalize vectors
            v1_norm = v1 / (np.linalg.norm(v1) + 1e-10)
            v2_norm = v2 / (np.linalg.norm(v2) + 1e-10)
            
            # Calculate angle (smaller angle = more prominent knee)
            cos_angle = np.clip(np.dot(v1_norm, v2_norm), -1, 1)
            angle = np.arccos(cos_angle)
            
            knee_scores[sorted_indices[i]] = np.pi - angle
            
        # Handle boundaries
        knee_scores[sorted_indices[0]] = 0.5
        knee_scores[sorted_indices[-1]] = 0.5
        
        return knee_scores

# Test advanced bias strategies
print("\nTesting advanced bias strategies:")
print("-"*40)

bias_strategy = MultiObjectiveBiasStrategy()

# Set preference direction
bias_strategy.set_preference_direction([1, 1])  # Prefer balanced improvement

# Create test population
test_pop = pareto_with_bias[:50]  # Use subset of Pareto front

# Apply different bias strategies
strategies = ["knee", "compromise", "extreme"]
results = {}

for strategy in strategies:
    print(f"\nApplying {strategy} region bias...")
    biased_pop = test_pop.copy()
    bias_strengths = bias_strategy.apply_region_bias(biased_pop, strategy)
    
    objectives = np.array([ind.objectives for ind in biased_pop])
    results[strategy] = {
        'objectives': objectives,
        'bias_strengths': bias_strengths
    }
    
    print(f"  Average bias strength: {np.mean(bias_strengths):.3f}")
    print(f"  Strong bias solutions: {np.sum(bias_strengths > 0.7)}")

# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (strategy, result) in enumerate(results.items()):
    ax = axes[idx]
    
    # Plot solutions colored by bias strength
    scatter = ax.scatter(result['objectives'][:, 0], result['objectives'][:, 1], 
                        c=result['bias_strengths'], cmap='viridis', 
                        s=50, alpha=0.7)
    
    # Add true Pareto front
    ax.plot(true_f1, true_f2, 'r--', linewidth=2, alpha=0.5, label='True front')
    
    ax.set_xlabel('f1')
    ax.set_ylabel('f2')
    ax.set_title(f'{strategy.capitalize()} Region Bias')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    
    # Add colorbar
    plt.colorbar(scatter, ax=ax, label='Bias Strength')

plt.tight_layout()
plt.show()

## Summary

### NSGA-II Algorithm Key Points:

1. **Core Mechanisms**
   - Fast non-dominated sorting: Hierarchical management of solution quality
   - Crowding distance calculation: Maintains solution diversity
   - Elitism preservation: Ensures optimal solutions are not lost

2. **Algorithm Flow**
   - Initialize population
   - Evolution loop (selection, crossover, mutation)
   - Environmental selection (merge parent and offspring)
   - Output Pareto front

3. **Bias Integration Benefits**
   - Accelerated convergence toward preferred regions
   - Guided search based on problem knowledge
   - Adaptive adjustment of search intensity
   - Better exploration-exploitation balance

4. **Application Scenarios**
   - Engineering design optimization
   - Resource allocation
   - Investment portfolio optimization
   - Any scenario requiring balancing multiple objectives

NSGA-II is a classic multi-objective optimization algorithm. When integrated with the NSGABLACK bias system, it becomes even more powerful, providing guided search toward user-preferred solutions while maintaining diversity and convergence.

## Next Tutorial

In the next tutorial, we'll explore [Bayesian Optimization and Intelligent Bias](05_Bayesian_Optimization_and_Intelligent_Bias.ipynb).